# 🎬 TD: Système de Recommandation de Films
## Semaine 4 - Classification Complète

**Objectif:** Prédire si un client va aimer un film en utilisant ses caractéristiques et historique.

### 📋 Ce que vous allez apprendre:
1. Prétraitement complet des données
2. Gestion des valeurs manquantes
3. Détection et traitement des outliers
4. Encodage de variables catégorielles
5. Normalisation
6. Classification avec plusieurs algorithmes
7. Évaluation et comparaison

## 📊 Étape 1: Chargement et Exploration des Données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Charger les données
df = pd.read_csv('data/clients_films.csv')

# Aperçu
print("📋 Aperçu des données:")
print(df.head())
print(f"\n📏 Dimensions: {df.shape}")
print(f"\n🔍 Types de données:")
print(df.dtypes)

### 🔍 Analyse exploratoire

In [ ]:
# Informations générales
print("ℹ️ Informations du dataset:")
df.info()

# Statistiques descriptives
print("\n📊 Statistiques descriptives:")
print(df.describe())

# Distribution de la variable cible
print("\n🎯 Distribution de la variable cible:")
print(df['Aime_Film'].value_counts())
print(df['Aime_Film'].value_counts(normalize=True))

## 🔧 Étape 2: Identification des Problèmes

In [ ]:
# ❌ Valeurs manquantes
print("❌ VALEURS MANQUANTES:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Nombre': missing, 'Pourcentage': missing_pct})
print(missing_df[missing_df['Nombre'] > 0])

# Visualisation
plt.figure(figsize=(10, 4))
missing_df[missing_df['Nombre'] > 0]['Nombre'].plot(kind='bar', color='coral')
plt.title('Valeurs manquantes par colonne')
plt.ylabel('Nombre de valeurs manquantes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 📈 Détection des Outliers

In [ ]:
# Variables numériques
numeric_cols = ['Age', 'Montant_Depense', 'Nb_Films_Vus', 
                'Temps_Visionnage_Minutes', 'Note_Moyenne', 'Frequence_Connexion_Jours']

print("⚠️ DÉTECTION DES OUTLIERS (Z-Score > 3):")
for col in numeric_cols:
    if df[col].notna().sum() > 0:
        z_scores = np.abs(stats.zscore(df[col].dropna()))
        outliers = (z_scores > 3).sum()
        print(f"{col}: {outliers} outliers")

# Boxplots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
    axes[i].tick_params(labelbottom=False)

plt.suptitle('Boxplots - Détection des Outliers', fontsize=16)
plt.tight_layout()
plt.show()

## 🛠️ Étape 3: Prétraitement des Données

### 3.1 Gestion des Valeurs Manquantes

In [ ]:
from sklearn.impute import SimpleImputer

# Solution complète fournie
# Créer une copie pour le prétraitement
df_clean = df.copy()

# Imputation Age avec la médiane (robuste aux outliers)
age_imputer = SimpleImputer(strategy='median')
df_clean['Age'] = age_imputer.fit_transform(df_clean[['Age']])

# Imputation Note_Moyenne avec la moyenne
note_imputer = SimpleImputer(strategy='mean')
df_clean['Note_Moyenne'] = note_imputer.fit_transform(df_clean[['Note_Moyenne']])

# Genre_Prefere: mode (valeur la plus fréquente)
genre_imputer = SimpleImputer(strategy='most_frequent')
df_clean['Genre_Prefere'] = genre_imputer.fit_transform(df_clean[['Genre_Prefere']]).ravel()

# Vérification
print("✅ Valeurs manquantes après imputation:")
print(df_clean.isnull().sum())

### 3.2 Traitement des Outliers

In [ ]:
# Méthode IQR pour capping

def cap_outliers_iqr(df, column):
    """Cap les outliers avec la méthode IQR"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    
    return df

# Appliquer sur les colonnes avec outliers
outlier_cols = ['Montant_Depense', 'Temps_Visionnage_Minutes']

for col in outlier_cols:
    print(f"📊 Avant capping {col}: min={df_clean[col].min():.2f}, max={df_clean[col].max():.2f}")
    df_clean = cap_outliers_iqr(df_clean, col)
    print(f"✅ Après capping {col}: min={df_clean[col].min():.2f}, max={df_clean[col].max():.2f}\n")

### 3.3 Encodage des Variables Catégorielles

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Solution fournie

# 1. Label Encoding pour Niveau_Engagement (ORDINAL)
le = LabelEncoder()
# Définir l'ordre: Faible < Moyen < Élevé
engagement_map = {'Faible': 0, 'Moyen': 1, 'Élevé': 2}
df_clean['Niveau_Engagement_Encoded'] = df_clean['Niveau_Engagement'].map(engagement_map)

# 2. One-Hot Encoding pour variables NOMINALES
df_clean = pd.get_dummies(df_clean, columns=['Genre_Prefere', 'Type_Abonnement'], 
                          prefix=['Genre', 'Abonnement'])

# 3. Encoder la variable cible
df_clean['Aime_Film_Encoded'] = (df_clean['Aime_Film'] == 'Oui').astype(int)

print("✅ Encodage terminé!")
print(f"📊 Nouvelles dimensions: {df_clean.shape}")
print(f"\n🏷️ Nouvelles colonnes:")
print(df_clean.columns.tolist())

### 3.4 Sélection des Features et Normalisation

In [ ]:
from sklearn.preprocessing import StandardScaler

# Sélectionner les features (X) et la cible (y)
# Exclure: ID_Client, colonnes originales encodées, et la cible
cols_to_drop = ['ID_Client', 'Niveau_Engagement', 'Aime_Film', 'Aime_Film_Encoded']

X = df_clean.drop(columns=cols_to_drop)
y = df_clean['Aime_Film_Encoded']

print(f"📊 Features (X): {X.shape}")
print(f"🎯 Cible (y): {y.shape}")
print(f"\n✨ Features utilisées:")
print(X.columns.tolist())

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\n✅ Normalisation terminée!")
print(X_scaled.describe())

## 🔀 Étape 4: Split Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

# Solution fournie
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Train set: {X_train.shape}")
print(f"📊 Test set: {X_test.shape}")
print(f"\n🎯 Distribution train:")
print(y_train.value_counts(normalize=True))
print(f"\n🎯 Distribution test:")
print(y_test.value_counts(normalize=True))

## 🤖 Étape 5: Modèles de Classification

### 5.1 K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Tester différentes valeurs de K
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Prédictions
y_pred_knn = knn.predict(X_test)

# Évaluation
print("🎯 KNN - Résultats:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.3f}")
print("\n📊 Rapport de classification:")
print(classification_report(y_test, y_pred_knn, target_names=['Non', 'Oui']))

# Matrice de confusion
cm_knn = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non', 'Oui'], yticklabels=['Non', 'Oui'])
plt.title('Matrice de Confusion - KNN')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.show()

### 5.2 Régression Logistique

In [ ]:
from sklearn.linear_model import LogisticRegression

# Solution fournie
logreg = LogisticRegression(random_state=42, max_iter=1000)
logreg.fit(X_train, y_train)

# Prédictions
y_pred_lr = logreg.predict(X_test)
y_proba_lr = logreg.predict_proba(X_test)[:, 1]

# Évaluation
print("📈 Régression Logistique - Résultats:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print("\n📊 Rapport de classification:")
print(classification_report(y_test, y_pred_lr, target_names=['Non', 'Oui']))

# Matrice de confusion
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Non', 'Oui'], yticklabels=['Non', 'Oui'])
plt.title('Matrice de Confusion - Régression Logistique')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.show()

### 5.3 Arbre de Décision

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Solution fournie
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

# Prédictions
y_pred_tree = tree.predict(X_test)

# Évaluation
print("🌳 Arbre de Décision - Résultats:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_tree):.3f}")
print("\n📊 Rapport de classification:")
print(classification_report(y_test, y_pred_tree, target_names=['Non', 'Oui']))

# Matrice de confusion
cm_tree = confusion_matrix(y_test, y_pred_tree)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Non', 'Oui'], yticklabels=['Non', 'Oui'])
plt.title('Matrice de Confusion - Arbre de Décision')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.show()

# Visualisation de l'arbre
plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=X.columns, class_names=['Non', 'Oui'], 
          filled=True, rounded=True, fontsize=10)
plt.title('Arbre de Décision - Visualisation')
plt.show()

## 📊 Étape 6: Comparaison des Modèles

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Calculer toutes les métriques
models = {
    'KNN': y_pred_knn,
    'Régression Logistique': y_pred_lr,
    'Arbre de Décision': y_pred_tree
}

results = []
for name, y_pred in models.items():
    results.append({
        'Modèle': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)
print("🏆 COMPARAISON DES MODÈLES:")
print(results_df.to_string(index=False))

# Visualisation
results_df.set_index('Modèle').plot(kind='bar', figsize=(12, 6), rot=0)
plt.title('Comparaison des Performances', fontsize=16)
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Meilleur modèle
best_model = results_df.loc[results_df['F1-Score'].idxmax(), 'Modèle']
best_f1 = results_df['F1-Score'].max()
print(f"\n🥇 Meilleur modèle: {best_model} (F1-Score: {best_f1:.3f})")

## 🎯 Étape 7: Prédictions sur Nouveaux Clients

In [ ]:
# Créer un nouveau client fictif
nouveau_client = pd.DataFrame({
    'Age': [28],
    'Montant_Depense': [120.0],
    'Nb_Films_Vus': [35],
    'Temps_Visionnage_Minutes': [4500],
    'Note_Moyenne': [4.2],
    'Frequence_Connexion_Jours': [3.5],
    'Niveau_Engagement_Encoded': [2],  # Élevé
    'Genre_Action': [1],
    'Genre_Comédie': [0],
    'Genre_Drame': [0],
    'Genre_Horreur': [0],
    'Genre_Romance': [0],
    'Genre_Science-Fiction': [0],
    'Abonnement_Gratuit': [0],
    'Abonnement_Premium': [1],
    'Abonnement_Standard': [0]
})

# Normaliser avec le même scaler
nouveau_client_scaled = scaler.transform(nouveau_client)

# Prédictions avec les 3 modèles
print("🔮 PRÉDICTIONS POUR LE NOUVEAU CLIENT:")
print("\nCaractéristiques:")
print("- Âge: 28 ans")
print("- Dépense: 120€")
print("- Films vus: 35")
print("- Note moyenne: 4.2/5")
print("- Engagement: Élevé")
print("- Genre préféré: Action")
print("- Abonnement: Premium\n")

pred_knn = knn.predict(nouveau_client_scaled)[0]
pred_lr = logreg.predict(nouveau_client_scaled)[0]
proba_lr = logreg.predict_proba(nouveau_client_scaled)[0][1]
pred_tree = tree.predict(nouveau_client_scaled)[0]

print(f"KNN: {'✅ Oui' if pred_knn == 1 else '❌ Non'}")
print(f"Régression Logistique: {'✅ Oui' if pred_lr == 1 else '❌ Non'} (probabilité: {proba_lr:.1%})")
print(f"Arbre de Décision: {'✅ Oui' if pred_tree == 1 else '❌ Non'}")

# Vote majoritaire
vote = sum([pred_knn, pred_lr, pred_tree])
final_pred = "Oui" if vote >= 2 else "Non"
print(f"\n🗳️ Vote majoritaire: {final_pred}")

## 📝 Exercices Supplémentaires

### Exercice 1: Optimisation de K pour KNN
Testez différentes valeurs de K (3, 5, 7, 9, 11) et trouvez la meilleure.

### Exercice 2: Feature Importance
Avec l'arbre de décision, affichez l'importance des features.

### Exercice 3: Courbe ROC
Tracez la courbe ROC pour la régression logistique.

### Exercice 4: Cross-Validation
Utilisez la validation croisée pour évaluer plus robustement les modèles.

In [ ]:
# EXERCICE 1: Optimisation K
# Solution fournie
k_values = [3, 5, 7, 9, 11]
k_scores = []

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train, y_train)
    score = knn_temp.score(X_test, y_test)
    k_scores.append(score)
    print(f"K={k}: Accuracy={score:.3f}")

# Visualisation
plt.figure(figsize=(10, 6))
plt.plot(k_values, k_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('K (nombre de voisins)')
plt.ylabel('Accuracy')
plt.title('Optimisation de K pour KNN')
plt.grid(True, alpha=0.3)
plt.show()

best_k = k_values[np.argmax(k_scores)]
print(f"\n🏆 Meilleur K: {best_k} (Accuracy: {max(k_scores):.3f})")

In [ ]:
# EXERCICE 2: Feature Importance
# Solution fournie
importances = tree.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("🔝 TOP 10 Features les plus importantes:")
print(feature_importance_df.head(10))

# Visualisation
plt.figure(figsize=(12, 6))
feature_importance_df.head(10).plot(x='Feature', y='Importance', kind='barh')
plt.title('Importance des Features - Arbre de Décision')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 🎉 Conclusion

### Ce que vous avez appris:
✅ Prétraitement complet d'un dataset réel  
✅ Gestion des valeurs manquantes avec différentes stratégies  
✅ Détection et traitement des outliers  
✅ Encodage de variables catégorielles (One-Hot et Label)  
✅ Normalisation des données  
✅ Application de 3 algorithmes de classification  
✅ Évaluation avec métriques appropriées  
✅ Comparaison et sélection du meilleur modèle  

### Points clés à retenir:
- Le prétraitement est **crucial** (70% du temps en ML!)
- Toujours **visualiser** avant de traiter
- **Train/Test split** AVANT normalisation (éviter data leakage)
- Utiliser plusieurs **métriques** (pas juste Accuracy)
- Comparer plusieurs modèles et choisir selon le contexte

🚀 **Bravo! Vous êtes prêts pour des projets réels!**